# Constructing Graph Representation

The **graph representation** is the intermediate form between a circuit and the measurement angles the
machine is driven with: the operations have been laid out on the machine's two-dimensional lattice of
macronodes, but they are still named as operations rather than as angles.

You do not have to write one — a client compiles a circuit representation for you, and
{ref}`sec:graph-composing-shots` below is the one thing on this page that a circuit cannot express. Write
one when you want to control the layout, or to see what a conversion produced.

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

## Creating a graph representation

Constructing a quantum circuit in graph representation starts with creating an instance of the {py:class}`~claveles.graph.GraphRepr`.
Graph representation is created by specifying the following two parameters.

* {py:attr}`~claveles.graph.GraphRepr.n_local_macronodes` : the number of macronodes per step
* {py:attr}`~claveles.graph.GraphRepr.n_steps` : the number of steps

For example, graph representation with `n_local_macronodes=3` and `n_steps=4` can be created as follows.
Every macronode is initialized with an operation {py:class}`~claveles.graph.Wiring` which is configured to pass each mode straight through.

In [ ]:
from claveles.graph import GraphRepr

g = GraphRepr(3, 4)

You can visualize graph representation using {py:func}`~claveles.graph.visualize.make_figure`.
See [Visualizing Graph Representation](viz_graph_repr.ipynb) for detailed usage of {py:func}`~claveles.graph.visualize.make_figure`.

In [ ]:
from claveles.graph.visualize import make_figure

make_figure(g);

Every macronode has both a coordinate and an index.

* The coordinate is `(h, w)`, running from `(0, 0)` at the top left to
  `(n_local_macronodes - 1, n_steps - 1)` at the bottom right.
* The index numbers the macronodes down each column in turn, so macronode `k` and macronode `k + 1` are
  vertically adjacent except where a column ends.

{py:meth}`~claveles.graph.GraphRepr.get_coord` converts an index to a coordinate and
{py:meth}`~claveles.graph.GraphRepr.get_index` a coordinate to an index.

```{image} _images/graph_repr_coord.svg
:alt: Macronode coordinates and indices
:width: 500px
```

Writing `N` for the number of local macronodes and `M` for the number of steps, macronode `k` has an edge
to macronode `k + 1` and to macronode `k + N`. Those two edges are what a mode is teleported along, which
is why every operation below names the mode arriving from `k - 1` and the one arriving from `k - N`.

Drawn in two dimensions, the `k -> k + 1` edge out of the bottom macronode of a column, that is out of
macronode `i * N - 1` for `i = 1, 2, ..., M - 1`, runs from the bottom of one column to the top of the
next. It is an ordinary edge in spite of how it looks: nothing about it is special.

In [ ]:
print(g.get_coord(0))
print(g.get_coord(11))
print(g.get_index(0, 0))
print(g.get_index(2, 3))

## Placing operations

{py:meth}`~claveles.graph.GraphRepr.place_operation` places an operation on a macronode. The operations
themselves live in {py:mod}`claveles.graph.ops`, and {ref}`Gates <sec:gates-in-graph-repr>` documents each of
them with the measurement angles it becomes.

Two things work differently here from the circuit representation.

* **A mode has to be initialized before anything can be applied to it**, with
  {py:class}`~claveles.graph.ops.Initialization`. There is no state to place; see
  {ref}`the graph.ops.Initialization entry <sec:gate-graph-repr-initialization>`.
* **Displacement is not an operation of its own.** It is carried by whichever operation the displaced
  mode arrives at, as one of two constructor arguments:

  | Argument | Displaces the mode teleported to macronode `k` from |
  | --- | --- |
  | {py:attr}`~claveles.graph.Operation.displacement_k_minus_1` | macronode `k - 1` |
  | {py:attr}`~claveles.graph.Operation.displacement_k_minus_n` | macronode `k - n_local_macronodes` |

In [ ]:
from claveles.graph.constant import BLANK_MODE
from claveles.graph.ops import Initialization

g = GraphRepr(1, 2)
# initialized_modes=(BLANK_MODE, 0) means that the mode teleporting to the macronode `k + 1` is `BLANK_MODE`,
# and the mode teleporting to the macronode `k + N` is mode `0`.
# `BLANK_MODE` means that this mode is not the target of any operations.
g.place_operation(Initialization(macronode=(0, 0), theta=0, initialized_modes=(BLANK_MODE, 0)))
make_figure(g);

The circuit below can be implemented as follows:

```{image} _images/graph_repr_example_light.svg
:alt: graph_repr_example
:class: only-light
```
```{image} _images/graph_repr_example_dark.svg
:alt: graph_repr_example
:class: only-dark
```

In [ ]:
from numpy import pi

from claveles.graph.ops import Initialization, Measurement, PhaseRotation

g1 = GraphRepr(1, 3)

# Displacement is specified by `displacement_k_minus_n`.
g1.place_operation(Initialization(macronode=(0, 0), theta=0, initialized_modes=(BLANK_MODE, 0)))
g1.place_operation(PhaseRotation(macronode=(0, 1), phi=pi / 4, swap=False, displacement_k_minus_n=(-1, 0)))
g1.place_operation(Measurement(macronode=(0, 2), theta=pi / 2, displacement_k_minus_n=(0, 1)))

make_figure(g1, scale=4.0);

A graph representation allows an equivalent circuit to be implemented in multiple ways.
For example, the circuit from earlier can be implemented as follows.

In [ ]:
from numpy import pi

from claveles.graph import Wiring
from claveles.graph.ops import Initialization, Measurement, PhaseRotation

g2 = GraphRepr(2, 3)

g2.place_operation(Initialization(macronode=(0, 0), theta=0, initialized_modes=(BLANK_MODE, 0)))
g2.place_operation(PhaseRotation(macronode=(0, 1), phi=pi / 4, swap=True, displacement_k_minus_n=(-1, 0)))
g2.place_operation(Wiring(macronode=(1, 1), swap=True, displacement_k_minus_1=(0, 1)))
g2.place_operation(Measurement(macronode=(1, 2), theta=pi / 2))

make_figure(g2, scale=4.0);

The equivalence of the two graph representations `g1` and `g2`  can be confirmed by the fact that the lists of operations applied to the mode are identical.

In [ ]:
print(g1.calc_mode_operations(0))
print(g2.calc_mode_operations(0))

### Adding Feedforward

In the graph representation, feedforward allows the parameters of certain operations or displacements to be updated based on the measurement result of another mode.
To begin, we create the graph representation of the following two-mode system:


In [ ]:
from numpy import pi

from claveles.graph.ops import ControlledZ, Measurement, PhaseRotation

g = GraphRepr(3, 5)

g.place_operation(Initialization((1, 0), 0.0, (BLANK_MODE, 0)))
g.place_operation(PhaseRotation((1, 1), pi / 2, swap=False, displacement_k_minus_n=(1, -1)))
g.place_operation(Initialization((0, 2), 0.0, (1, BLANK_MODE)))
g.place_operation(ControlledZ((1, 2), 1, swap=True))
g.place_operation(Measurement((2, 2), 0))
g.place_operation(Measurement((1, 4), pi / 2))

# Check the operations applied to each mode
print(g.calc_mode_operations(0))
print(g.calc_mode_operations(1))

make_figure(g);

Suppose that a {py:class}`~claveles.graph.ops.PhaseRotation` and a displacement are applied to mode 1, depending on the
measurement result of mode 0. The following code adds feedforward to the graph representation accordingly:

In [ ]:
from claveles.feedforward import feedforward


@feedforward
def f1(x: float) -> float:
    return x * x


@feedforward
def f2(x: float) -> float:
    return x * x * x


x = g.get_mode_measured_value(0)
g.place_operation(PhaseRotation((1, 3), f1(x), swap=False, displacement_k_minus_n=(f1(x), f2(x))))

make_figure(g);

## Converting circuit representation to graph representation

```{image} _images/circuit_repr_sample_circuit_light.svg
:alt: sample circuit
:class: only-light
```
```{image} _images/circuit_repr_sample_circuit_dark.svg
:alt: sample circuit
:class: only-dark
```

Convert the above circuit representation to a graph representation.
This can be achieved as follows:

* Place operations that initialize the modes.
* Place each operation, excluding the displacements, on the macronodes
* Displace the modes that are teleporting

In [ ]:
from numpy import pi, sqrt

from claveles.graph.ops import ControlledZ, Measurement, PhaseRotation

g = GraphRepr(3, 4)

# The circuit leaves both modes open, and a conversion initializes an open mode with theta = -pi/2,
# which is squeezing angle 0.
g.place_operation(Initialization((1, 0), -pi / 2, (BLANK_MODE, 0)))
g.place_operation(PhaseRotation((1, 1), pi / 4, swap=False))
g.place_operation(Initialization((0, 2), -pi / 2, (1, BLANK_MODE)))

# The circuit displaces mode 0 only after rotating it, and a displacement is applied to a mode
# as it arrives, so the controlled-Z carries it rather than the rotation. The circuit asks for the
# amplitude alpha = 1 - i; a graph displacement is the shift itself, which is sqrt(2) times that.
g.place_operation(ControlledZ((1, 2), 1, swap=True, displacement_k_minus_n=(sqrt(2), -sqrt(2))))

# Mode 1 ends at macronode (1, 3) and mode 0 at (2, 2), so those carry the measurements. A graph
# Measurement(theta) is the circuit's MeasureHomodyne(phi) at theta = pi/2 - phi: mode 0, measured at
# phi = pi/2, takes theta = 0, and mode 1, measured at phi = 0, takes theta = pi/2.
g.place_operation(Measurement((1, 3), pi / 2))
g.place_operation(Measurement((2, 2), 0))

# Check the operations applied to each mode
print(g.calc_mode_operations(0))
print(g.calc_mode_operations(1))

In [ ]:
make_figure(g, show_displacement=True);

{py:mod}`claveles.graph.convert` converts a circuit representation to a graph representation with
{py:class}`~claveles.graph.convert.CGConverter`. The settings handed to it choose the strategy used to place
the operations, and there are two to choose between.

* {py:class}`~claveles.graph.convert.GreedyConvertSettings` : Place the operations with a greedy algorithm. It runs very quickly but tends to use a large number of macronodes.
* {py:class}`~claveles.graph.convert.BeamSearchConvertSettings` : Place the operations with a beam search. It also runs quickly, and uses fewer macronodes.

{py:obj}`~claveles.graph.convert.GreedyConverter` and {py:obj}`~claveles.graph.convert.BeamSearchConverter`
are other names for that one converter, so that a call can name the strategy alongside its settings.
The conversion then reads as follows:

In [ ]:
from numpy import pi

from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import ControlledZ, Displacement, MeasureHomodyne, PhaseRotation

c = CircuitRepr("multiple_gates")
c.Q(0) | PhaseRotation(phi=pi / 4) | Displacement(1 - 1j)
c.Q(0, 1) | ControlledZ(g=1)
c.Q(0) | MeasureHomodyne(phi=pi / 2)
c.Q(1) | MeasureHomodyne(phi=0)

c

In [ ]:
from claveles.graph.convert import BeamSearchConverter, BeamSearchConvertSettings

g = BeamSearchConverter(BeamSearchConvertSettings(n_local_macronodes=3, beam_width=10)).convert(c)

# Check the operations applied to each mode
print(g.calc_mode_operations(0))
print(g.calc_mode_operations(1))

In [ ]:
make_figure(g);

(sec:graph-composing-shots)=

## Packing several shots into one graph

A macronode lattice is usually far larger than one program needs, and the machine runs a whole
lattice per execution either way. {py:mod}`claveles.graph.compose` fills the spare room with further
copies of the same graph, so that one execution yields several shots.

{py:func}`~claveles.graph.compose.compose_into_composite_graph` takes the graph and a
{py:class}`~claveles.graph.compose.ComposeSettings` saying how many shots are wanted and how large the
composite may be:


In [ ]:
from claveles.graph.compose import ComposeSettings, compose_into_composite_graph

original = GraphRepr(2, 2)
original.place_operation(Initialization((0, 0), 0.0, (BLANK_MODE, 0)))
original.place_operation(Measurement((0, 1), 0.0))

settings = ComposeSettings(n_shots=10, n_local_macronodes=6, max_steps=20)
composite = compose_into_composite_graph(original, settings)

print("composite size  =", composite.graph.n_local_macronodes, "x", composite.graph.n_steps)
print("shots per run   =", composite.mapping.n_shots)
print("runs needed     =", composite.n_executions)


The three counts answer different questions and are deliberately named apart:

* {py:attr}`ComposeSettings.n_shots <claveles.graph.compose.ComposeSettings.n_shots>` is how many shots you
  asked for.
* {py:attr}`~claveles.graph.compose.MappingInfo.n_shots` is how many copies fit into one composite.
* {py:attr}`~claveles.graph.compose.CompositeGraph.n_executions` is how many times the composite has to
  be run to reach the number asked for.

Copies are separated by rows and columns of {py:class}`~claveles.graph.ops.Measurement` macronodes so
that no mode crosses from one copy into the next. A graph that passes a mode from the bottom of a
column to the top of the next cannot be stacked vertically and is only placed side by side.

The composite is an ordinary {py:class}`~claveles.graph.GraphRepr`, so it is executed like any other.
{py:func}`~claveles.graph.compose.decompose_composite_graph_result` then splits its result back into
one shot per copy, using the mapping:


In [ ]:
from claveles.client import SimulatorClient
from claveles.execute import execute
from claveles.graph.compose import decompose_composite_graph_result

composite_result = execute(composite.graph, SimulatorClient(n_shots=composite.n_executions))
original_result = decompose_composite_graph_result(
    composite_result.execution_result,
    composite.mapping,
    original,
)

print("shots recovered =", original_result.n_shots)
print("first shot      =", original_result.get_shot_measured_value(0).coord_list())


Each recovered shot is indexed by the *original* graph's coordinates, not the composite's, so the
result reads as though the original had been run on its own.
{py:func}`~claveles.machinery.compose.decompose_composite_machinery_result` does the same for a
composite executed as a machinery representation.


(sec:graph-initial-states)=

## Initial states

Every mode in this representation starts as a **squeezed state**, produced by an initialization
macronode; {ref}`the graph.ops.Initialization entry <sec:gate-graph-repr-initialization>` explains why a measurement macronode and an
initialization macronode are the same thing.

Unlike the circuit representation, which lets you place a state on a mode (see
{ref}`sec:circuit-placing-states`), this representation has no way to express a state at all. What can be
set, and where:

| | Set by |
| --- | --- |
| **Squeezing level** | Not expressible here. On the simulator it is the client option {py:attr}`~claveles.client.simulator_client.SimulatorClient.resource_squeezing_level`; on the machine it is fixed by the hardware. |
| **Squeezing angle** | The {py:class}`~claveles.graph.ops.Initialization` operation's `theta` parameter: the mode is initialized with {ref}`squeezing angle <sec:squeezing-level-and-squeezing-angle>` $\theta + \pi/2$. |

So a mode reaches this representation in one of two forms, and a conversion refuses anything else:

- **left open** — it starts from the state the machine supplies. This is the usual thing to write.
- **carrying {py:meth}`~claveles.circuit.state.QuantumState.hardware_squeezed`** — its `phi` sets the
  initialization's `theta` to $\phi - \pi/2$, which is how the squeezing angle of a mode is chosen.

Any other placed state raises `ValueError`, naming the modes and the positions of the placements to drop
({py:meth}`~claveles.circuit.CircuitRepr.remove` takes them). A
{py:meth}`~claveles.circuit.state.QuantumState.hardware_squeezed` placed alongside is not one of them, so it
keeps its angle. `post_select` is refused for the same reason — a graph has no field for a fixed outcome;
see {ref}`sec:support-features`.

```{seealso}
{ref}`sec:execution-resource-and-initialized-states` for the values the machine uses and
{ref}`sec:simulation-resource-and-initialized-states` for the simulator's.
```